# MatSafe Stage 5 -- Embed & Index (Colab / GPU)

Moved here from local CPU-only execution: a local test showed 3 models x 6 chunk sets (55,597 total chunks) would take an estimated 6-8 hours on CPU. Same logic as `encode_chunks.py` + `build_faiss_index.py` in the repo, adapted to run on GPU here and download results back down.

**Steps:**
1. Runtime -> Change runtime type -> GPU (T4 is fine), before running anything below.
2. Run all cells in order.
3. Cell 3 will prompt a file upload -- upload `chunks_tagged_for_colab.zip` from your local `Maternal_RAG_Corpus/` folder.
4. The last cell downloads `indexes_from_colab.zip` -- unzip it into your local `Maternal_RAG_Corpus/indexes/` folder to merge back with the repo.

In [ ]:
!pip install -q sentence-transformers faiss-cpu huggingface_hub

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Set Runtime -> Change runtime type -> GPU, then re-run this cell.'

In [ ]:
from google.colab import files
import zipfile, os

print('Upload chunks_tagged_for_colab.zip (from your local Maternal_RAG_Corpus/ folder):')
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name) as zf:
    zf.extractall('.')
CHUNKS_TAGGED = 'chunks_tagged'
print('extracted:', os.listdir(CHUNKS_TAGGED))

In [ ]:
# Same MODELS list, prefix conventions, and manifest schema as encode_chunks.py
# in the repo -- kept identical so results merge cleanly with the local pipeline.
# CORPUS_GIT_COMMIT: the commit chunks_tagged/ was frozen at when this zip was
# made (see CORPUS_LOG.md's freeze declaration). Update if the corpus is
# re-frozen before re-running this notebook.
CORPUS_GIT_COMMIT = 'bff28ca1b087413096eb370d9280d39d093d2b06'

MODELS = [
    {
        'key': 'minilm',
        'hf_name': 'sentence-transformers/all-MiniLM-L6-v2',
        'passage_prefix': '',
        'query_prefix': '',
        'tier': 'small_general',
    },
    {
        'key': 'e5-base',
        'hf_name': 'intfloat/e5-base-v2',
        'passage_prefix': 'passage: ',
        'query_prefix': 'query: ',
        'tier': 'strong_general',
    },
    {
        'key': 'pubmedbert-msmarco',
        'hf_name': 'pritamdeka/S-PubMedBert-MS-MARCO',
        'passage_prefix': '',
        'query_prefix': '',
        'tier': 'biomedical',
    },
]

In [ ]:
import csv, json, time
import numpy as np
from huggingface_hub import HfApi
from sentence_transformers import SentenceTransformer

INDEX_ROOT = 'indexes'
os.makedirs(INDEX_ROOT, exist_ok=True)

def load_chunk_set(fname):
    with open(os.path.join(CHUNKS_TAGGED, fname), encoding='utf-8') as f:
        return list(csv.DictReader(f))

chunk_set_files = sorted(os.listdir(CHUNKS_TAGGED))
built, skipped = [], []

for model_spec in MODELS:
    print(f"\n=== {model_spec['key']} ({model_spec['hf_name']}) ===")
    t0 = time.time()
    try:
        model = SentenceTransformer(model_spec['hf_name'], device='cuda')
    except Exception as e:
        print(f"  SKIPPING {model_spec['key']}: failed to load ({e})")
        skipped.append(model_spec['key'])
        continue
    load_s = round(time.time() - t0, 1)
    print(f'  loaded on GPU in {load_s}s')

    try:
        model_sha = HfApi().model_info(model_spec['hf_name']).sha
    except Exception:
        model_sha = None

    for fname in chunk_set_files:
        chunk_set_name = fname[:-4]
        rows = load_chunk_set(fname)
        out_dir = os.path.join(INDEX_ROOT, f"{model_spec['key']}__{chunk_set_name}")
        os.makedirs(out_dir, exist_ok=True)

        texts = [model_spec['passage_prefix'] + r['text'] for r in rows]
        chunk_ids = [r['chunk_id'] for r in rows]

        t0 = time.time()
        embeddings = model.encode(texts, batch_size=128, show_progress_bar=False, convert_to_numpy=True)
        encode_s = round(time.time() - t0, 1)

        norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
        norms[norms == 0] = 1.0
        embeddings = (embeddings / norms).astype('float32')

        np.save(os.path.join(out_dir, 'embeddings.npy'), embeddings)
        with open(os.path.join(out_dir, 'chunk_ids.json'), 'w', encoding='utf-8') as f:
            json.dump(chunk_ids, f)

        manifest = {
            'model_hf_name': model_spec['hf_name'],
            'model_hf_commit_sha': model_sha,
            'model_tier': model_spec['tier'],
            'embedding_dimension': embeddings.shape[1],
            'normalization': 'L2-normalized vectors, inner product (FAISS IndexFlatIP) == cosine similarity',
            'passage_prefix_applied': model_spec['passage_prefix'] or '(none)',
            'query_prefix_to_apply_at_retrieval': model_spec['query_prefix'] or '(none)',
            'chunk_set': chunk_set_name,
            'n_chunks': len(rows),
            'corpus_git_commit': CORPUS_GIT_COMMIT,
            'encode_time_seconds': encode_s,
            'model_load_time_seconds': load_s,
            'encoded_on': 'Colab GPU',
        }
        with open(os.path.join(out_dir, 'manifest.json'), 'w', encoding='utf-8') as f:
            json.dump(manifest, f, indent=2)

        print(f'    {len(rows)} chunks -> dim {embeddings.shape[1]}, encode {encode_s}s')
        built.append(manifest)

print(f'\nEncoded {len(built)} (model, chunk_set) pairs. Skipped: {skipped or "none"}')

In [ ]:
# FAISS indexing -- CPU is fine here, index build for a few thousand
# vectors is fast regardless of GPU/CPU.
import faiss

for name in sorted(os.listdir(INDEX_ROOT)):
    out_dir = os.path.join(INDEX_ROOT, name)
    emb_path = os.path.join(out_dir, 'embeddings.npy')
    if not os.path.isfile(emb_path):
        continue
    embeddings = np.load(emb_path)
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)
    faiss.write_index(index, os.path.join(out_dir, 'index.faiss'))
    print(f'  {name}: {index.ntotal} vectors indexed')

print('\nFAISS indexing complete.')

In [ ]:
import shutil
shutil.make_archive('indexes_from_colab', 'zip', INDEX_ROOT)
files.download('indexes_from_colab.zip')
print('Downloaded indexes_from_colab.zip -- unzip into your local Maternal_RAG_Corpus/indexes/ folder.')